In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load files
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")

# Parse timestamps to UTC datetimes as required
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregation
metric_agg_full = (
    metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        mean='mean',
        P50=lambda s: s.quantile(0.5),
        P90=lambda s: s.quantile(0.9),
        P95=lambda s: s.quantile(0.95),
    )
    .reset_index()
)
metric_agg_full[['mean','P50','P90','P95']] = metric_agg_full[['mean','P50','P90','P95']].astype(float)
metric_top20 = metric_agg_full.sort_values('count', ascending=False).head(20)[['cmdb_id','kpi_name','count','mean','P50','P90','P95']]
metric_top20[['mean','P50','P90','P95']] = metric_top20[['mean','P50','P90','P95']].round(6)

# 2) trace.csv aggregation
trace_agg_full = (
    trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        count='count',
        mean='mean',
        P50=lambda s: s.quantile(0.5),
        P90=lambda s: s.quantile(0.9),
        P95=lambda s: s.quantile(0.95),
    )
    .reset_index()
)
trace_agg_full[['mean','P50','P90','P95']] = trace_agg_full[['mean','P50','P90','P95']].astype(float)
trace_top20 = trace_agg_full.sort_values('count', ascending=False).head(20)[['cmdb_id','trace_name','count','mean','P50','P90','P95']]
trace_top20[['mean','P50','P90','P95']] = trace_top20[['mean','P50','P90','P95']].round(6)

# 3) log.csv aggregation
log_agg_full = (
    log
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(
        count='count',
        mean='mean',
        P50=lambda s: s.quantile(0.5),
        P90=lambda s: s.quantile(0.9),
        P95=lambda s: s.quantile(0.95),
    )
    .reset_index()
)
log_agg_full[['mean','P50','P90','P95']] = log_agg_full[['mean','P50','P90','P95']].astype(float)
log_top20 = log_agg_full.sort_values('count', ascending=False).head(20)[['cmdb_id','log_name','count','mean','P50','P90','P95']]
log_top20[['mean','P50','P90','P95']] = log_top20[['mean','P50','P90','P95']].round(6)

# Unique cmdb_id counts per file
unique_counts = pd.DataFrame({
    'file': ['metric.csv', 'trace.csv', 'log.csv'],
    'unique_cmdb_id_count': [metric['cmdb_id'].nunique(), trace['cmdb_id'].nunique(), log['cmdb_id'].nunique()]
})

# Display compact results (top-20 aggregated tables and unique cmdb_id counts)
metric_top20, trace_top20, log_top20, unique_counts

```
Out[1]:
```
summary = (
    "Summary of full-series aggregations (plain English):\n\n"
    "Unique components observed: metric.csv: 69 cmdb_id values; trace.csv: 28 cmdb_id values; log.csv: 46 cmdb_id values.\n\n"
    "Metric highlights:\n"
    "- The most notable CPU hotspot is ts-travel-service (mean CPU ≈ 23.76, P50 ≈ 25.89, P90 ≈ 38.94, P95 ≈ 39.31), followed by ts-travel2-service (mean ≈ 9.67, P90 ≈ 15.49).\n"
    "- Large memory values appear for ts-travel2-mongo (mean ≈ 69,406,080) and ts-voucher-service (mean ≈ 38,260,810).\n"
    "- Significant disk I/O is seen for ts-travel-service (mean ≈ 630,997) and ts-travel2-mongo (mean ≈ 387,793).\n"
    "- Higher latencies are present in travel services: ts-travel2-service latency (P90 ≈ 0.49 s, P95 ≈ 0.83 s) and ts-travel-service latency (P90 ≈ 0.44 s, P95 ≈ 0.93 s).\n\n"
    "Trace highlights:\n"
    "- High throughput (row_count) for ts-travel2-service traces (from_ts-travel2-service.row_count mean ≈ 2546.28, P50 ≈ 2630, P90 ≈ 2827.6), indicating heavy traffic through that service path.\n"
    "- Many trace error_rate measures reported 0.0 in the top aggregated entries, i.e., no obvious trace error spikes in those top groups.\n\n"
    "Log highlights:\n"
    "- Error counts in the top rows are 0.0 for many services (no large per-minute error_count observed in the aggregated top entries).\n"
    "- High log throughput seen for ts-basic-service (log.row_count mean ≈ 2408.28), ts-route-service (≈ 986), ts-auth-service (≈ 322), and ts-order-service (≈ 362).\n\n"
    "Overall conclusion:\n"
    "- There are no large error spikes visible in the top log/trace error_count or trace.error_rate aggregates.\n"
    "- Metric signals point to resource and latency pressure on travel-related components: ts-travel-service (very high CPU and elevated latency) and ts-travel2-service (elevated CPU and latency), with ts-travel2-mongo showing high memory, disk I/O and very high throughput. These components are the primary hotspots to investigate further.\n\n"
    "Full aggregated DataFrames are available in the kernel as: metric_agg_full, trace_agg_full, log_agg_full."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count          mean           P50           P90           P95
360           ts-voucher-service      socket     25  4.017143e+00  4.000000e+00  4.000000e+00  4.000000e+00
359           ts-voucher-service         mem     25  3.826081e+07  3.809792e+07  3.809905e+07  3.809957e+07
358           ts-voucher-service      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
2    ts-admin-basic-info-service         cpu     25  1.034095e+00  8.798080e-01  1.668343e+00  1.821055e+00
3    ts-admin-basic-info-service      diskio     25  5.441621e+04  5.148092e+04  6.944016e+04  7.722159e+04
4    ts-admin-basic-info-service  latency-50     25  1.855400e-02  1.753600e-02  1.937600e-02  2.014600e-02
5    ts-admin-basic-info-service  latency-90     25  2.872000e-02  2.356500e-02  3.743400e-02  5.914400e-02
319             ts-travel2-mongo         cpu     25  6.905180e-01  6.888900e-01  7.270820e-01  7.399240e-01
320             ts-travel2-mongo      diskio     25  3.877927e+05  3.704328e+05  4.529764e+05  4.602669e+05
321             ts-travel2-mongo         mem     25  6.940608e+07  6.894449e+07  7.083561e+07  7.085786e+07
322             ts-travel2-mongo      socket     25  6.000000e+00  6.000000e+00  6.000000e+00  6.000000e+00
323           ts-travel2-service         cpu     25  9.670702e+00  7.459200e+00  1.548601e+01  1.721441e+01
324           ts-travel2-service      diskio     25  5.502295e+04  5.415254e+04  6.519517e+04  7.016591e+04
325           ts-travel2-service  latency-50     25  2.520300e-02  1.977200e-02  2.199800e-02  2.902800e-02
326           ts-travel2-service  latency-90     25  4.915030e-01  3.872630e-01  6.133360e-01  8.260510e-01
311       ts-travel-plan-service      socket     25  3.045333e+00  3.000000e+00  3.226667e+00  3.260000e+00
312            ts-travel-service         cpu     25  2.376196e+01  2.589469e+01  3.893641e+01  3.930977e+01
313            ts-travel-service      diskio     25  6.309966e+05  5.298068e+04  6.628602e+04  7.069616e+04
314            ts-travel-service  latency-50     25  6.107600e-02  6.491800e-02  1.335020e-01  1.574300e-01
315            ts-travel-service  latency-90     25  4.374370e-01  4.158520e-01  8.142480e-01  9.293280e-01,                 cmdb_id                                         trace_name  count         mean          P50          P90          P95
783     ts-user-service                 trace.to_ts-user-service.row_count     25    12.080000    12.000000    16.000000    17.600000
0                  root  trace.to_ts-admin-basic-info-service.duration_...     25     0.015275     0.013786     0.018602     0.025402
1                  root  trace.to_ts-admin-basic-info-service.duration_p95     25     0.021593     0.017795     0.026000     0.049493
2                  root    trace.to_ts-admin-basic-info-service.error_rate     25     0.000000     0.000000     0.000000     0.000000
3                  root     trace.to_ts-admin-basic-info-service.row_count     25    17.680000    18.000000    20.000000    20.000000
782     ts-user-service                trace.to_ts-user-service.error_rate     25     0.000000     0.000000     0.000000     0.000000
781     ts-user-service              trace.to_ts-user-service.duration_p95     25     0.003560     0.003373     0.004060     0.004425
780     ts-user-service             trace.to_ts-user-service.duration_mean     25     0.002324     0.002280     0.002606     0.002723
779     ts-user-service               trace.from_ts-user-service.row_count     25    12.080000    12.000000    16.000000    17.600000
778     ts-user-service              trace.from_ts-user-service.error_rate     25     0.000000     0.000000     0.000000     0.000000
738  ts-travel2-service              trace.from_ts-seat-service.error_rate     25     0.000000     0.000000     0.000000     0.000000
739  ts-travel2-service               trace.from_ts-seat-service.row_count     25   313.520000   326.000000   351.200000   366.400000
740  ts-travel2-service        trace.from_ts-travel2-service.duration_mean     25     0.013843     0.011468     0.019771     0.024331
741  ts-travel2-service         trace.from_ts-travel2-service.duration_p95     25     0.060503     0.050789     0.086368     0.097903
742  ts-travel2-service           trace.from_ts-travel2-service.error_rate     25     0.000000     0.000000     0.000000     0.000000
743  ts-travel2-service            trace.from_ts-travel2-service.row_count     25  2546.280000  2630.000000  2827.600000  2939.600000
744  ts-travel2-service      trace.to_ts-order-other-service.duration_mean     25     0.006980     0.006109     0.009934     0.010582
745  ts-travel2-service       trace.to_ts-order-other-service.duration_p95     25     0.014460     0.008644     0.021802     0.050195
732  ts-travel2-service  trace.from_ts-preserve-other-service.duration_...     25     0.415929     0.326669     0.670619     0.714787
733  ts-travel2-service  trace.from_ts-preserve-other-service.duration_p95     25     0.501235     0.365031     0.950906     1.132951,                         cmdb_id         log_name  count     mean     P50     P90     P95
0   ts-admin-basic-info-service  log.error_count     25     0.00     0.0     0.0     0.0
1   ts-admin-basic-info-service    log.row_count     25    26.52    27.0    30.0    30.0
6          ts-assurance-service  log.error_count     25     0.00     0.0     0.0     0.0
7          ts-assurance-service    log.row_count     25     8.92     9.0    12.6    13.0
10              ts-auth-service  log.error_count     25     0.00     0.0     0.0     0.0
11              ts-auth-service    log.row_count     25   322.08   336.0   353.2   355.6
26          ts-contacts-service  log.error_count     25     0.00     0.0     0.0     0.0
27          ts-contacts-service    log.row_count     25    36.28    36.0    48.0    52.8
23           ts-consign-service    log.row_count     25    49.28    49.0    67.2    75.6
22           ts-consign-service  log.error_count     25     0.00     0.0     0.0     0.0
13             ts-basic-service    log.row_count     25  2408.28  2478.0  2716.6  2775.8
12             ts-basic-service  log.error_count     25     0.00     0.0     0.0     0.0
63             ts-price-service    log.row_count     25   263.68   272.0   298.0   307.6
62             ts-price-service  log.error_count     25     0.00     0.0     0.0     0.0
66             ts-route-service  log.error_count     25     0.00     0.0     0.0     0.0
67             ts-route-service    log.row_count     25   986.00  1022.0  1088.4  1101.2
56    ts-preserve-other-service  log.error_count     25     0.00     0.0     0.0     0.0
57    ts-preserve-other-service    log.row_count     25    74.08    57.0   129.0   139.8
49             ts-order-service    log.row_count     25   361.76   383.0   444.4   447.6
48             ts-order-service  log.error_count     25     0.00     0.0     0.0     0.0,          file  unique_cmdb_id_count
0  metric.csv                    69
1   trace.csv                    28
2     log.csv                    46)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed to UTC datetime
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Filter metric.csv to the five components
components = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']
metric_candidates = metric[metric['cmdb_id'].isin(components)].copy()

# 2) Compute global aggregates (full series for these components)
# Include P05 as well because we may need it for low-is-bad KPIs
metric_candidates_agg = (
    metric_candidates
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        mean='mean',
        P05=lambda s: s.quantile(0.05),
        P50=lambda s: s.quantile(0.5),
        P90=lambda s: s.quantile(0.9),
        P95=lambda s: s.quantile(0.95),
    )
    .reset_index()
)
# Round numeric columns for compactness
for col in ['mean','P05','P50','P90','P95']:
    metric_candidates_agg[col] = metric_candidates_agg[col].astype(float).round(6)

# Prepare the compact table to return (top 50) with required columns
metric_candidates_agg_top50 = metric_candidates_agg.sort_values('count', ascending=False)[
    ['cmdb_id','kpi_name','count','mean','P50','P90','P95']
].head(50)

# 3) Filter original metric rows for incident window (UTC)
incident_start = pd.to_datetime("2024-01-25 01:57:04", utc=True)
incident_end   = pd.to_datetime("2024-01-25 02:27:04", utc=True)

metric_candidates_incident = metric_candidates[
    (metric_candidates['timestamp'] >= incident_start) &
    (metric_candidates['timestamp'] <= incident_end)
].copy()

# 4) For each (cmdb_id, kpi_name) in the incident window compute incident stats and anomalies vs global thresholds
# Determine low-is-bad KPIs by keywords in kpi_name
def is_low_is_bad(kpi_name):
    kn = str(kpi_name).lower()
    for kw in ('success', 'ss', 'rate'):
        if kw in kn:
            return True
    return False

# Build a lookup of thresholds from global aggregates (metric_candidates_agg)
agg_lookup = metric_candidates_agg.set_index(['cmdb_id','kpi_name'])

rows = []
groups = metric_candidates_incident.groupby(['cmdb_id','kpi_name'])
for (cmdb_id, kpi_name), g in groups:
    total_points = int(g.shape[0])
    # Get global thresholds; if missing, skip (shouldn't happen)
    try:
        global_P95 = float(agg_lookup.loc[(cmdb_id,kpi_name),'P95'])
        global_P90 = float(agg_lookup.loc[(cmdb_id,kpi_name),'P90'])
        global_P05 = float(agg_lookup.loc[(cmdb_id,kpi_name),'P05'])
    except Exception:
        # fallback: compute from full series (rare)
        sfull = metric_candidates[(metric_candidates['cmdb_id']==cmdb_id)&(metric_candidates['kpi_name']==kpi_name)]['value']
        global_P95 = float(sfull.quantile(0.95)) if len(sfull) else np.nan
        global_P90 = float(sfull.quantile(0.9)) if len(sfull) else np.nan
        global_P05 = float(sfull.quantile(0.05)) if len(sfull) else np.nan

    low_is_bad = is_low_is_bad(kpi_name)
    if low_is_bad:
        anomaly_mask = g['value'] < global_P05
        threshold_used = global_P05
    else:
        anomaly_mask = g['value'] >= global_P95
        threshold_used = global_P95

    anomaly_count = int(anomaly_mask.sum())
    earliest_anom_ts = pd.NaT
    if anomaly_count > 0:
        earliest_anom_ts = g.loc[anomaly_mask, 'timestamp'].min()
    # incident max/min values
    incident_max = float(g['value'].max())
    incident_min = float(g['value'].min())

    rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'incident_count': total_points,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': pd.to_datetime(earliest_anom_ts).tz_convert('UTC') if pd.notnull(earliest_anom_ts) else pd.NaT,
        'threshold_used': threshold_used,
        'incident_max_value': round(incident_max,6),
        'incident_min_value': round(incident_min,6),
        'low_is_bad': low_is_bad
    })

anomaly_df_p95 = pd.DataFrame(rows)
# Compact formatting: convert timestamps to ISO strings or empty
anomaly_df_p95['earliest_anomaly_timestamp'] = anomaly_df_p95['earliest_anomaly_timestamp'].apply(
    lambda x: x.isoformat() if pd.notnull(x) else None
)

# Limit to top 50 rows sorted by anomaly_count desc then incident_count desc
anomaly_df_p95 = anomaly_df_p95.sort_values(['anomaly_count','incident_count'], ascending=[False,False]).head(50)

# 5) If no anomalies by >=P95, compute >=P90 breaches similarly
total_anomalies_p95 = int(anomaly_df_p95['anomaly_count'].sum())
if total_anomalies_p95 == 0:
    # compute using P90 threshold for high-is-bad, or P10 for low-is-bad? The instruction says relax to >=P90, so only relax high-is-bad to P90.
    rows90 = []
    for (cmdb_id, kpi_name), g in groups:
        total_points = int(g.shape[0])
        global_P90 = float(agg_lookup.loc[(cmdb_id,kpi_name),'P90'])
        low_is_bad = is_low_is_bad(kpi_name)
        if low_is_bad:
            # keep original low-is-bad rule (P05); do not relax low-is-bad unless specified
            anomaly_mask = g['value'] < float(agg_lookup.loc[(cmdb_id,kpi_name),'P05'])
            threshold_used = float(agg_lookup.loc[(cmdb_id,kpi_name),'P05'])
        else:
            anomaly_mask = g['value'] >= global_P90
            threshold_used = global_P90

        anomaly_count = int(anomaly_mask.sum())
        earliest_anom_ts = pd.NaT
        if anomaly_count > 0:
            earliest_anom_ts = g.loc[anomaly_mask, 'timestamp'].min()
        incident_max = float(g['value'].max())
        incident_min = float(g['value'].min())

        rows90.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'incident_count': total_points,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': pd.to_datetime(earliest_anom_ts).tz_convert('UTC') if pd.notnull(earliest_anom_ts) else pd.NaT,
            'threshold_used': threshold_used,
            'incident_max_value': round(incident_max,6),
            'incident_min_value': round(incident_min,6),
            'low_is_bad': low_is_bad
        })
    anomaly_df_p90 = pd.DataFrame(rows90)
    anomaly_df_p90['earliest_anomaly_timestamp'] = anomaly_df_p90['earliest_anomaly_timestamp'].apply(
        lambda x: x.isoformat() if pd.notnull(x) else None
    )
    anomaly_df_p90 = anomaly_df_p90.sort_values(['anomaly_count','incident_count'], ascending=[False,False]).head(50)
else:
    anomaly_df_p90 = pd.DataFrame()  # empty, not used

# Final outputs (compact)
metric_candidates_agg_top50, anomaly_df_p95 if total_anomalies_p95>0 else anomaly_df_p90, metric_candidates_incident.shape

```
Out[2]:
```
summary = (
    "Summary (plain English):\n\n"
    "- I filtered metric.csv to the five services you specified and examined the full historic distributions and the 30-minute incident window (2024-01-25 01:57:04 to 2024-01-25 02:27:04 UTC).\n\n"
    "- Incident data size: 875 metric rows for those five services in the incident window.\n\n"
    "- Global aggregates: there are 35 (cmdb_id, kpi_name) groups for these five services (stored in metric_candidates_agg in the kernel).\n\n"
    "- Anomalies vs global P95: multiple KPIs across these services breached their global P95 thresholds during the incident window. Breaches are found for common KPIs including cpu, mem, diskio, latency-50, latency-90, socket, and workload.\n\n"
    "- Typical breach pattern: most KPI groups that breached had small counts of anomalous points (anomaly_count = 2 for many groups). The earliest anomaly timestamps are around 2024-01-25T02:00:00+00:00 for several KPIs; other earliest breach times appear between ~02:00 and ~02:23 UTC depending on KPI.\n\n"
    "- Notable services to investigate first: ts-auth-service and ts-travel-service show prominent breaches across CPU, memory, disk I/O and latency KPIs. These services are likely hotspots during the incident window.\n\n"
    "- All intermediate results are retained in the kernel as metric_candidates_agg (global aggregates) and metric_candidates_incident (incident-window rows).\n\n"
    "Recommendation: start troubleshooting on ts-auth-service and ts-travel-service (review CPU, memory, disk I/O and latency metrics and related traces/logs) because they show repeated >=P95 breaches in the incident window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  count          mean           P50           P90           P95
0     ts-auth-service         cpu     25  1.635983e+01  1.614595e+01  1.728231e+01  1.763516e+01
1     ts-auth-service      diskio     25  5.359639e+04  5.210962e+04  6.496097e+04  6.547133e+04
2     ts-auth-service  latency-50     25  3.052690e-01  2.750890e-01  3.997570e-01  4.759120e-01
3     ts-auth-service  latency-90     25  9.226440e-01  8.550000e-01  1.322968e+00  1.535866e+00
4     ts-auth-service         mem     25  2.368162e+08  2.380325e+08  2.417791e+08  2.420164e+08
..                ...         ...    ...           ...           ...           ...           ...
30  ts-travel-service  latency-50     25  6.107600e-02  6.491800e-02  1.335020e-01  1.574300e-01
31  ts-travel-service  latency-90     25  4.374370e-01  4.158520e-01  8.142480e-01  9.293280e-01
32  ts-travel-service         mem     25  2.869352e+08  2.900329e+08  2.953530e+08  2.962490e+08
33  ts-travel-service      socket     25  2.397629e+01  2.953333e+01  3.160000e+01  3.214000e+01
34  ts-travel-service    workload     25  8.831858e+00  8.765450e+00  1.001663e+01  1.034825e+01

[35 rows x 7 columns],               cmdb_id    kpi_name  incident_count  anomaly_count earliest_anomaly_timestamp  threshold_used  incident_max_value  incident_min_value  low_is_bad
0     ts-auth-service         cpu              25              2  2024-01-25T02:00:00+00:00    1.763516e+01        2.388308e+01        1.413524e+01       False
1     ts-auth-service      diskio              25              2  2024-01-25T02:02:00+00:00    6.547133e+04        7.224747e+04        3.882972e+04       False
2     ts-auth-service  latency-50              25              2  2024-01-25T02:00:00+00:00    4.759120e-01        6.492380e-01        2.195250e-01       False
3     ts-auth-service  latency-90              25              2  2024-01-25T02:00:00+00:00    1.535866e+00        1.769711e+00        6.480190e-01       False
4     ts-auth-service         mem              25              2  2024-01-25T02:23:00+00:00    2.420164e+08        2.422415e+08        2.253910e+08       False
..                ...         ...             ...            ...                        ...             ...                 ...                 ...         ...
30  ts-travel-service  latency-50              25              2  2024-01-25T02:13:00+00:00    1.574300e-01        1.579530e-01        1.871300e-02       False
31  ts-travel-service  latency-90              25              2  2024-01-25T02:13:00+00:00    9.293280e-01        1.019544e+00        9.289200e-02       False
32  ts-travel-service         mem              25              2  2024-01-25T02:19:00+00:00    2.962490e+08        2.964489e+08        2.593887e+08       False
33  ts-travel-service      socket              25              2  2024-01-25T02:14:00+00:00    3.214000e+01        3.588333e+01        1.460714e+01       False
34  ts-travel-service    workload              25              2  2024-01-25T02:02:00+00:00    1.034825e+01        1.068777e+01        6.516700e+00       False

[35 rows x 9 columns], (875, 4))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing 'trace' if present; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed to UTC datetime
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# 1) Filter trace.csv to the five components
components = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']
trace_candidates = trace[trace['cmdb_id'].isin(components)].copy()

# 2) For each (cmdb_id, trace_name) compute global aggregates (include P05 for potential 'success' metrics)
trace_candidates_agg = (
    trace_candidates
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        count='count',
        mean='mean',
        P05=lambda s: s.quantile(0.05),
        P50=lambda s: s.quantile(0.5),
        P90=lambda s: s.quantile(0.9),
        P95=lambda s: s.quantile(0.95),
    )
    .reset_index()
)
# Round numeric columns for compactness
for col in ['mean','P05','P50','P90','P95']:
    trace_candidates_agg[col] = trace_candidates_agg[col].astype(float).round(6)

# Compact top-50 table to return with requested columns
trace_candidates_agg_top50 = trace_candidates_agg.sort_values('count', ascending=False)[
    ['cmdb_id','trace_name','count','mean','P50','P90','P95']
].head(50)

# 3) Filter original trace rows for the incident window (UTC)
incident_start = pd.to_datetime("2024-01-25 01:57:04", utc=True)
incident_end   = pd.to_datetime("2024-01-25 02:27:04", utc=True)

trace_candidates_incident = trace_candidates[
    (trace_candidates['timestamp'] >= incident_start) &
    (trace_candidates['timestamp'] <= incident_end)
].copy()

# 4) For each (cmdb_id, trace_name) in incident window compute incident_count, anomaly_count, earliest_anomaly_timestamp, threshold_used, incident_max_value
# Helper to determine special handling: 'success' (lower is bad), 'error_rate' (higher is bad). Default: higher is bad.
def is_low_is_bad(trace_name):
    tn = str(trace_name).lower()
    return 'success' in tn  # lower is bad for success metrics

# Build lookup from full aggregates (global thresholds computed on full series)
agg_lookup = trace_candidates_agg.set_index(['cmdb_id','trace_name'])

rows = []
groups = trace_candidates_incident.groupby(['cmdb_id','trace_name'])
for (cmdb_id, trace_name), g in groups:
    total_points = int(g.shape[0])
    try:
        global_P95 = float(agg_lookup.loc[(cmdb_id, trace_name), 'P95'])
        global_P90 = float(agg_lookup.loc[(cmdb_id, trace_name), 'P90'])
        global_P05 = float(agg_lookup.loc[(cmdb_id, trace_name), 'P05'])
    except Exception:
        # fallback to compute from full series for safety
        sfull = trace_candidates[(trace_candidates['cmdb_id']==cmdb_id)&(trace_candidates['trace_name']==trace_name)]['value']
        global_P95 = float(sfull.quantile(0.95)) if len(sfull) else np.nan
        global_P90 = float(sfull.quantile(0.9)) if len(sfull) else np.nan
        global_P05 = float(sfull.quantile(0.05)) if len(sfull) else np.nan

    low_is_bad = is_low_is_bad(trace_name)
    if low_is_bad:
        # anomaly when value < P05
        anomaly_mask = g['value'] < global_P05
        threshold_used = global_P05
    else:
        # default higher is worse (including error_rate)
        anomaly_mask = g['value'] > global_P95
        threshold_used = global_P95

    anomaly_count = int(anomaly_mask.sum())
    earliest_anom_ts = pd.NaT
    if anomaly_count > 0:
        earliest_anom_ts = g.loc[anomaly_mask, 'timestamp'].min()
    incident_max = float(g['value'].max())

    rows.append({
        'cmdb_id': cmdb_id,
        'trace_name': trace_name,
        'incident_count': total_points,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': pd.to_datetime(earliest_anom_ts).tz_convert('UTC') if pd.notnull(earliest_anom_ts) else pd.NaT,
        'threshold_used': threshold_used,
        'incident_max_value': round(incident_max,6),
        'low_is_bad': low_is_bad
    })

anomaly_df = pd.DataFrame(rows)
# Format earliest_anomaly_timestamp for compact display
anomaly_df['earliest_anomaly_timestamp'] = anomaly_df['earliest_anomaly_timestamp'].apply(
    lambda x: x.isoformat() if pd.notnull(x) else None
)

# Sort by anomaly_count desc then incident_count desc and limit to top 50
anomaly_df_top50 = anomaly_df.sort_values(['anomaly_count','incident_count'], ascending=[False,False]).head(50)

# Final outputs (compact): trace_candidates_agg_top50 and anomaly_df_top50
trace_candidates_agg_top50, anomaly_df_top50

```
Out[3]:
```
summary = (
    "Trace analysis summary (plain English):\n\n"
    "- I computed global trace aggregates and inspected the incident window 2024-01-25 01:57:04 → 2024-01-25 02:27:04 UTC (results stored in trace_candidates_agg and trace_candidates_incident).\n"
    "- Multiple trace metrics across the five services exceeded their global P95 thresholds during the incident window. These breaches are small in count per series (typically 2 points) but consistent across many trace groups.\n"
    "- The earliest anomalies appear around 2024-01-25T02:00:00Z (several services) with other earliest breaches around 02:02–02:13Z depending on the trace.\n"
    "- ts-auth-service stands out: several trace.duration and trace.row_count series exceeded P95 (duration spikes and increased row_count), indicating latency and load spikes involving auth paths.\n"
    "- ts-train-service also shows P95 breaches in duration and row_count traces, suggesting elevated load/latency on train-related calls.\n"
    "- trace.error_rate series in the top aggregates are generally 0.0, so these anomalies are latency/throughput spikes rather than obvious increases in trace error rates.\n"
    "- Recommendation: investigate ts-auth-service first (latency and traffic spikes), then inspect ts-train-service and any travel-related call paths. Correlate with metrics and logs for CPU/memory/diskIO and request patterns at ~02:00–02:15 UTC.\n\n"
    "Note: detailed aggregated tables are available in the kernel as trace_candidates_agg and the incident rows as trace_candidates_incident."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id                                  trace_name  count        mean         P50         P90         P95
0    ts-auth-service               trace.from_root.duration_mean     25    0.383907    0.362054    0.487814    0.525321
1    ts-auth-service                trace.from_root.duration_p95     25    0.917303    0.840779    1.362055    1.563921
2    ts-auth-service                  trace.from_root.error_rate     25    0.000000    0.000000    0.000000    0.000000
3    ts-auth-service                   trace.from_root.row_count     25   80.560000   84.000000   88.600000   89.000000
4    ts-auth-service    trace.from_ts-auth-service.duration_mean     25    0.004004    0.003403    0.005489    0.005742
..               ...                                         ...    ...         ...         ...         ...         ...
61  ts-route-service  trace.from_ts-travel2-service.duration_p95     25    0.007295    0.004661    0.016953    0.021646
62  ts-route-service    trace.from_ts-travel2-service.error_rate     25    0.000000    0.000000    0.000000    0.000000
63  ts-route-service     trace.from_ts-travel2-service.row_count     25  575.640000  595.000000  636.000000  656.800000
48  ts-route-service   trace.from_ts-basic-service.duration_mean     25    0.004567    0.003613    0.008250    0.009430
49  ts-route-service    trace.from_ts-basic-service.duration_p95     25    0.011748    0.004538    0.045359    0.052807

[50 rows x 7 columns],              cmdb_id                                  trace_name  incident_count  anomaly_count earliest_anomaly_timestamp  threshold_used  incident_max_value  low_is_bad
0    ts-auth-service               trace.from_root.duration_mean              25              2  2024-01-25T02:00:00+00:00        0.525321            0.702277       False
1    ts-auth-service                trace.from_root.duration_p95              25              2  2024-01-25T02:02:00+00:00        1.563921            1.627580       False
4    ts-auth-service    trace.from_ts-auth-service.duration_mean              25              2  2024-01-25T02:00:00+00:00        0.005742            0.014505       False
5    ts-auth-service     trace.from_ts-auth-service.duration_p95              25              2  2024-01-25T02:00:00+00:00        0.010579            0.094235       False
7    ts-auth-service        trace.from_ts-auth-service.row_count              25              2  2024-01-25T02:11:00+00:00      355.600000          370.000000       False
..               ...                                         ...             ...            ...                        ...             ...                 ...         ...
71  ts-train-service       trace.from_ts-basic-service.row_count              25              2  2024-01-25T02:01:00+00:00      153.800000          159.000000       False
72  ts-train-service   trace.from_ts-train-service.duration_mean              25              2  2024-01-25T02:03:00+00:00        0.001489            0.001548       False
73  ts-train-service    trace.from_ts-train-service.duration_p95              25              2  2024-01-25T02:00:00+00:00        0.001714            0.001957       False
75  ts-train-service       trace.from_ts-train-service.row_count              25              2  2024-01-25T02:01:00+00:00     2754.400000         2868.000000       False
76  ts-train-service  trace.from_ts-travel-service.duration_mean              25              2  2024-01-25T02:00:00+00:00        0.008473            0.009396       False

[50 rows x 8 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing 'log' and load 'error_logs' if not present
try:
    log
except NameError:
    log = pd.read_csv("log.csv")
try:
    error_logs
except NameError:
    error_logs = pd.read_csv("error_logs.csv")

# Ensure timestamps are parsed to UTC datetimes
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Target components and incident window (UTC)
components = ['ts-auth-service', 'ts-travel-service']
incident_start = pd.to_datetime("2024-01-25 01:57:04", utc=True)
incident_end   = pd.to_datetime("2024-01-25 02:27:04", utc=True)

# 1) Filter full series to target components and compute global aggregates
log_candidates = log[log['cmdb_id'].isin(components)].copy()

log_candidates_agg = (
    log_candidates
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(
        count='count',
        mean='mean',
        P50=lambda s: s.quantile(0.5),
        P90=lambda s: s.quantile(0.9),
        P95=lambda s: s.quantile(0.95),
    )
    .reset_index()
)

# Round numeric columns for compact display
for col in ['mean','P50','P90','P95']:
    log_candidates_agg[col] = log_candidates_agg[col].astype(float).round(6)

# Compact top-20 aggregated table
log_candidates_agg_top20 = log_candidates_agg.sort_values('count', ascending=False)[
    ['cmdb_id','log_name','count','mean','P50','P90','P95']
].head(20)

# 2) Filter log rows to incident window for same services
log_candidates_incident = log_candidates[
    (log_candidates['timestamp'] >= incident_start) &
    (log_candidates['timestamp'] <= incident_end)
].copy()

# Build lookup of global P95 thresholds (computed on full series)
agg_lookup = log_candidates_agg.set_index(['cmdb_id','log_name'])

rows = []
groups = log_candidates_incident.groupby(['cmdb_id','log_name'])
for (cmdb_id, log_name), g in groups:
    incident_count = int(g.shape[0])
    # global threshold lookup (computed before incident filtering)
    try:
        global_P95 = float(agg_lookup.loc[(cmdb_id, log_name), 'P95'])
    except Exception:
        # fallback to compute from the full series subset if lookup missing
        sfull = log_candidates[(log_candidates['cmdb_id']==cmdb_id)&(log_candidates['log_name']==log_name)]['value']
        global_P95 = float(sfull.quantile(0.95)) if len(sfull) else np.nan

    anomaly_mask = g['value'] > global_P95
    anomaly_count = int(anomaly_mask.sum())
    earliest_anom_ts = pd.NaT
    if anomaly_count > 0:
        earliest_anom_ts = g.loc[anomaly_mask, 'timestamp'].min()
    incident_max = float(g['value'].max())
    incident_min = float(g['value'].min())

    rows.append({
        'cmdb_id': cmdb_id,
        'log_name': log_name,
        'incident_count': incident_count,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': pd.to_datetime(earliest_anom_ts).tz_convert('UTC') if pd.notnull(earliest_anom_ts) else pd.NaT,
        'threshold_used': global_P95,
        'incident_max_value': round(incident_max,6),
        'incident_min_value': round(incident_min,6)
    })

log_anomaly_df = pd.DataFrame(rows)
# Format earliest_anomaly_timestamp as ISO strings or None
log_anomaly_df['earliest_anomaly_timestamp'] = log_anomaly_df['earliest_anomaly_timestamp'].apply(
    lambda x: x.isoformat() if pd.notnull(x) else None
)

log_anomaly_top20 = log_anomaly_df.sort_values(['anomaly_count','incident_count'], ascending=[False,False]).head(20)

# 3) Filter error_logs.csv for the two services and incident window
error_logs_filtered = error_logs[
    (error_logs['cmdb_id'].isin(components)) &
    (error_logs['timestamp'] >= incident_start) &
    (error_logs['timestamp'] <= incident_end)
].copy()

# Prepare compact view (up to 20 rows) with timestamp ISO, cmdb_id, message
error_logs_filtered_view = error_logs_filtered.loc[:, ['timestamp','cmdb_id','message']].copy()
error_logs_filtered_view['timestamp'] = error_logs_filtered_view['timestamp'].dt.tz_convert('UTC').apply(lambda x: x.isoformat())

error_logs_count = int(error_logs_filtered.shape[0])

# Final compact outputs (stored variables: log_candidates_agg, log_candidates_incident, error_logs_filtered)
log_candidates_agg_top20, log_anomaly_top20, error_logs_filtered_view.head(20), error_logs_count

```
Out[4]:
```
summary = (
    "Log & error-log summary (plain English):\n\n"
    "- Global log aggregates (full series):\n"
    "  * ts-auth-service: log.row_count mean ≈ 322.08, P95 = 355.6; log.error_count is 0.0 (no global error spikes).\n"
    "  * ts-travel-service: log.row_count mean ≈ 1617.88, P95 = 1922.2; log.error_count is 0.0 (no global error spikes).\n\n"
    "- Incident-window findings (2024-01-25 01:57:04 → 2024-01-25 02:27:04 UTC):\n"
    "  * Both services had log.row_count values exceeding their global P95 during the incident window (anomaly_count = 2 for each).\n"
    "    - ts-travel-service: earliest breach at 2024-01-25T02:01:00+00:00, incident_max log.row_count = 2058.0 (threshold P95 = 1922.2).\n"
    "    - ts-auth-service: earliest breach at 2024-01-25T02:11:00+00:00, incident_max log.row_count = 370.0 (threshold P95 = 355.6).\n"
    "  * log.error_count remained 0 (no error-count spikes) for both services during the window.\n\n"
    "- Error log messages: no matching error_logs rows were found for ts-auth-service or ts-travel-service in the incident window (0 rows).\n\n"
    "Conclusion / recommendation:\n"
    "- The issue appears related to traffic/load spikes (increased row_count) on ts-travel-service (early, ~02:01 UTC) and ts-auth-service (later, ~02:11 UTC), not to elevated error rates or recorded error-log messages.\n"
    "- Investigate the source of the increased request volume and correlate with the metric and trace anomalies (CPU, latency, traces) already observed for these services.\n\n"
    "Notes: Aggregated tables and filtered incident data are stored in the kernel as: log_candidates_agg, log_candidates_incident, error_logs_filtered."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  count     mean     P50     P90     P95
0    ts-auth-service  log.error_count     25     0.00     0.0     0.0     0.0
1    ts-auth-service    log.row_count     25   322.08   336.0   353.2   355.6
2  ts-travel-service  log.error_count     25     0.00     0.0     0.0     0.0
3  ts-travel-service    log.row_count     25  1617.88  1716.0  1886.6  1922.2,              cmdb_id         log_name  incident_count  anomaly_count earliest_anomaly_timestamp  threshold_used  incident_max_value  incident_min_value
1    ts-auth-service    log.row_count              25              2  2024-01-25T02:11:00+00:00           355.6               370.0                20.0
3  ts-travel-service    log.row_count              25              2  2024-01-25T02:01:00+00:00          1922.2              2058.0               202.0
0    ts-auth-service  log.error_count              25              0                       None             0.0                 0.0                 0.0
2  ts-travel-service  log.error_count              25              0                       None             0.0                 0.0                 0.0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [], 0)```
```

In [ ]:
# In[5]:

import re
import pandas as pd
import numpy as np

# Reuse existing variables from kernel: trace_candidates_incident, trace_candidates_agg
# If they're missing, raise informative error
if 'trace_candidates_incident' not in globals() or 'trace_candidates_agg' not in globals():
    raise RuntimeError("Required variables trace_candidates_incident and trace_candidates_agg not found in kernel.")

# 1) Extract rows whose trace_name contains 'to_ts-' or 'from_ts-'
mask_to_from = trace_candidates_incident['trace_name'].str.contains(r'\b(?:to_ts-|from_ts-)', regex=True, na=False)
parsed_trace_refs = trace_candidates_incident[mask_to_from].copy()

# Parse referenced service name (capture token after to_ts- or from_ts-)
parsed_trace_refs['referenced_service'] = parsed_trace_refs['trace_name'].str.extract(r'(?:to_ts-|from_ts-)([a-z0-9\-]+)', flags=re.IGNORECASE, expand=False)

# Normalize to lowercase
parsed_trace_refs['referenced_service'] = parsed_trace_refs['referenced_service'].str.lower()

# 2) Keep only referenced services in the candidate list
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']
parsed_trace_refs = parsed_trace_refs[parsed_trace_refs['referenced_service'].isin(candidates)].copy()

# Prepare lookup for global thresholds from trace_candidates_agg (index by caller cmdb_id and trace_name)
agg_lookup = trace_candidates_agg.set_index(['cmdb_id','trace_name'])

# Helper to check low-is-bad trace names
def is_low_is_bad_trace(name):
    return 'success' in str(name).lower()

# 3) Aggregate per (caller_cmdb_id, referenced_service, trace_name)
rows = []
grouped = parsed_trace_refs.groupby(['cmdb_id','referenced_service','trace_name'])
for (caller, callee, tname), g in grouped:
    incident_count = int(g.shape[0])
    # lookup global thresholds (computed from full series for caller,tname)
    try:
        global_P95 = float(agg_lookup.loc[(caller, tname), 'P95'])
        global_P05 = float(agg_lookup.loc[(caller, tname), 'P05'])
    except Exception:
        # fallback: try to find in trace_candidates_agg DataFrame
        tmp = trace_candidates_agg[(trace_candidates_agg['cmdb_id']==caller)&(trace_candidates_agg['trace_name']==tname)]
        if not tmp.empty:
            global_P95 = float(tmp['P95'].iloc[0])
            global_P05 = float(tmp['P05'].iloc[0])
        else:
            # final fallback: compute from all incident rows for that caller/tname
            sfull = parsed_trace_refs[(parsed_trace_refs['cmdb_id']==caller)&(parsed_trace_refs['trace_name']==tname)]['value']
            global_P95 = float(sfull.quantile(0.95)) if len(sfull) else np.nan
            global_P05 = float(sfull.quantile(0.05)) if len(sfull) else np.nan

    low_is_bad = is_low_is_bad_trace(tname)
    if low_is_bad:
        anomaly_mask = g['value'] < global_P05
        threshold_used = global_P05
    else:
        anomaly_mask = g['value'] > global_P95
        threshold_used = global_P95

    anomaly_count = int(anomaly_mask.sum())
    earliest_anom_ts = pd.NaT
    if anomaly_count > 0:
        earliest_anom_ts = g.loc[anomaly_mask, 'timestamp'].min()
    incident_max_value = float(g['value'].max())

    rows.append({
        'caller_cmdb_id': caller,
        'referenced_service': callee,
        'trace_name': tname,
        'incident_count': incident_count,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anom_ts,  # keep as datetime for now
        'threshold_used': threshold_used,
        'incident_max_value': round(incident_max_value,6)
    })

# Ensure DataFrame has expected columns even if no rows
if rows:
    paired_summary = pd.DataFrame(rows)
else:
    paired_summary = pd.DataFrame(columns=[
        'caller_cmdb_id','referenced_service','trace_name','incident_count',
        'anomaly_count','earliest_anomaly_timestamp','threshold_used','incident_max_value'
    ])

# Format earliest_anomaly_timestamp to ISO strings (or None)
if 'earliest_anomaly_timestamp' in paired_summary.columns:
    paired_summary['earliest_anomaly_timestamp'] = paired_summary['earliest_anomaly_timestamp'].apply(
        lambda x: x.tz_convert('UTC').isoformat() if pd.notnull(x) else None
    )

# 3) Sort and limit to top 50
paired_summary_top50 = paired_summary.sort_values(['anomaly_count','earliest_anomaly_timestamp'], ascending=[False, True]).head(50)

# 4) Aggregated caller->callee table: sum anomaly_count and earliest anomaly timestamp per pair
if not paired_summary.empty:
    tmp = paired_summary.copy()
    # convert earliest_anomaly_timestamp back to datetime for aggregation (coerce None)
    tmp['earliest_ts_dt'] = pd.to_datetime(tmp['earliest_anomaly_timestamp'], utc=True, errors='coerce')
    caller_callee_agg = tmp.groupby(['caller_cmdb_id','referenced_service']).agg(
        total_anomaly_count = ('anomaly_count','sum'),
        earliest_anomaly_timestamp = ('earliest_ts_dt','min')
    ).reset_index()
    # format timestamp ISO or None
    caller_callee_agg['earliest_anomaly_timestamp'] = caller_callee_agg['earliest_anomaly_timestamp'].apply(
        lambda x: x.isoformat() if pd.notnull(x) else None
    )
    caller_callee_agg_top20 = caller_callee_agg.sort_values(['total_anomaly_count','earliest_anomaly_timestamp'], ascending=[False, True]).head(20)
else:
    caller_callee_agg_top20 = pd.DataFrame(columns=['caller_cmdb_id','referenced_service','total_anomaly_count','earliest_anomaly_timestamp'])

# Outputs (compact tables)
paired_summary_top50, caller_callee_agg_top20

```
Out[5]:
```
Summary (plain English):

- I searched the incident-window traces for calls containing "to_ts-" or "from_ts-" and then parsed the downstream service names.
- No parsed referenced services matched the candidate list [ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service].
- As a result, the caller→callee detail table and the aggregated caller→callee anomaly table are both empty.

Possible reasons / next steps:
- The traces in the incident window may use different naming conventions (not containing "to_ts-" / "from_ts-") or reference services outside the candidate list.
- To proceed, inspect trace_candidates_incident.trace_name (e.g., trace_candidates_incident['trace_name'].unique() or .head(50)) to find actual naming patterns, then re-run parsing with the correct patterns or expand the candidate callee list.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [caller_cmdb_id, referenced_service, trace_name, incident_count, anomaly_count, earliest_anomaly_timestamp, threshold_used, incident_max_value]
Index: [], Empty DataFrame
Columns: [caller_cmdb_id, referenced_service, total_anomaly_count, earliest_anomaly_timestamp]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd

# Ensure required variable exists
if 'trace_candidates_incident' not in globals():
    raise RuntimeError("trace_candidates_incident not found in kernel. Please run prior trace filtering step.")

# Aggregate unique trace_name counts in the incident window
trace_name_counts = (
    trace_candidates_incident
    .groupby('trace_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
)

trace_name_unique_count = int(trace_candidates_incident['trace_name'].nunique())

# Return compact results
trace_name_counts, trace_name_unique_count

```
Out[6]:
```
top = trace_name_counts.head(5)
top_list = [f"{row['trace_name']}: {int(row['count'])}" for _, row in top.iterrows()]

summary = (
    f"Trace-name pattern summary for incident window:\n"
    f"- Total unique trace_name values: {trace_name_unique_count}.\n"
    f"- Top trace_name patterns (by count): " + "; ".join(top_list) + ".\n"
    f"- Observation: trace.from_ts-travel-service.* entries dominate (multiple entries with count=100), indicating heavy activity on ts-travel-service call paths during the incident window. Several other services (e.g., auth, train) show smaller, consistent counts (~25).\n"
    f"- Recommendation: prioritize investigation of ts-travel-service trace paths (row_count and duration metrics) as primary hotspots."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                    trace_name  count
51      trace.from_ts-travel-service.row_count    100
50     trace.from_ts-travel-service.error_rate    100
49   trace.from_ts-travel-service.duration_p95    100
48  trace.from_ts-travel-service.duration_mean    100
0                trace.from_root.duration_mean     75
..                                         ...    ...
45    trace.from_ts-train-service.duration_p95     25
46      trace.from_ts-train-service.error_rate     25
47       trace.from_ts-train-service.row_count     25
8     trace.from_ts-auth-service.duration_mean     25
9      trace.from_ts-auth-service.duration_p95     25

[50 rows x 2 columns], 88)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```